# Inhalte aus Metadaten extrahieren

### Benötigte Bibliotheken importieren:

In [2]:
from bs4 import BeautifulSoup as soup
import unicodedata
from lxml import etree
import pandas as pd

### Laden des XML-Files: 

In [3]:
# Laden der MARC-xml-Datei in ElementTree und umwandeln in eine Liste an Datensätzen: 
tree = etree.parse('20260706_MARC21-xml_sw=Dissertation.xml')
root = tree.getroot()
ns = {'marc': 'http://www.loc.gov/MARC21/slim'}
records = root.findall('.//marc:record', namespaces=ns)
print("Gefundene Records:", len(records))

Gefundene Records: 1171


### Erstellen einer Funktion für die Extraktion der gewünschten Informationen

Hier können die Felder entsprechend den eigenen Wünschen angepasst werden:

In [4]:
def parse_record(record):
    
    ns = {"marc": "http://www.loc.gov/MARC21/slim"} 
    
    def extract_text(xpath_query):
        fields = record.xpath(xpath_query, namespaces=ns)
        if fields:
            return "; ".join(field.text for field in fields if field.text)
        return None

    # Hier Felder anpassen: 
    # 1. Vor dem Gleichheitszeichen steht eine frei wählbare Bezeichnung der Information (es wird eine Variable erzeugt, bspw. mit dem Name "titel")
    # 2. Nach dem Gleichheitszeichen wird die oben angelegte Funktion zur Extraktion des Feldinhalts aus dem jeweiligen MARC12-xml-Feld aufgerufen. Hierfür 
    #        muss lediglich der Typ des Feldes (datafield oder controlfield), die Nummer des Feldes ("tag") und, wenn zutreffend, das zugehörige Unterfeld (subfield)
    #        angepasst werden. 
    
    idn = extract_text("marc:controlfield[@tag='001']")
    author = extract_text("marc:datafield[@tag='100']/marc:subfield[@code='a']")
    author2 = extract_text("marc:datafield[@tag='700']/marc:subfield[@code='a']")
    title = extract_text("marc:datafield[@tag='245']/marc:subfield[@code='a']")
    subtitle = extract_text("marc:datafield[@tag='245']/marc:subfield[@code='b']")
    statement_of_responsibility = extract_text("marc:datafield[@tag='245']/marc:subfield[@code='c']")
    publisher = extract_text("marc:datafield[@tag='264']/marc:subfield[@code='b']")
    place = extract_text("marc:datafield[@tag='264']/marc:subfield[@code='a']")
    pages = extract_text("marc:datafield[@tag='300']/marc:subfield[@code='a']")
    pub_year = extract_text("marc:datafield[@tag='264']/marc:subfield[@code='c']")
    #year_008 = extract_text("marc:controlfield[@tag='008']")[7:11]
    links = extract_text("marc:datafield[@tag='856']/marc:subfield[@code='u']")
    ddc = extract_text("marc:datafield[@tag='082']/marc:subfield[@code='a']")
    
    # 3. Zuletzt müssen neue Variablen hier ergänzt, entfernte gelöscht werden.
    #     Zum Löschen dafür jeweils den gesamten Teil zwischen zwei Kommata löschen, also bsw. "links":links um die Ausgabe von Links zu entfernen
    #     Zum Ergänzen einfach an gewünschter Stelle nach vorliegendem Schema neue Variablen ergänzen: Hier ist der erste Teil in "" wieder ein frei wählbarer
    #     Name (dies wird der Spaltenname in einer später auszugebenden Tabelle), nach dem Doppelpunkt folgt dann der zugehörigen eben oben definierte Variablenname:
    return {"idn": idn, "author": author, "author2": author2,"title": title, "subtitle": subtitle, "responsible":statement_of_responsibility, 
            "publisher": publisher, "place": place, "pages": pages, "pub_year":pub_year, "links":links, "ddc": ddc}

Aufrufen der Funktion, Übergabe der einzelnen Datensätze und Transformation in eine Tabelle: 

In [5]:
result = [parse_record(record) for record in records]
df = pd.DataFrame(result)
df

,idn,author,author2,title,subtitle,responsible,publisher,place,pages,pub_year,links,ddc
0,1395347972,"Feldmüller, Tim","Czulo, Oliver; Bubenhofer, Noah; Czulo, Oliver...",Das Framing von Extremismusvarianten im medi...,NaN,Tim Feldmüller ; Gutachter: Oliver Czulo ; Ol...,Leibniz-Institut für Deutsche Sprache (IDS); ...,Mannheim; Berlin,Online-Ressource,2026; 2026,https://doi.org/10.5281/zenodo.18274343; https...,401.4
1,1389623513,"Ebel, Simone","Mastalerz, Michael",Functionalized and Conformationally Stable Mon...,NaN,Simone Felicia Ebel ; Betreuer: Michael Mastalerz,Universitätsbibliothek Heidelberg,Heidelberg,Online-Ressource,2026,https://doi.org/10.11588/heidok.00038080; http...,547
2,1398116688,"Jeschke, Tanja",NaN,"Jeschke, T. (2026). Sänger, Maike (2023): Wis...",NaN,Tanja Jeschke,Leibniz-Institut für Deutsche Sprache (IDS); ...,Mannheim; Mannheim,Online-Ressource,2026; 2026,https://doi.org/10.21248/jfml.2026.106; https:...,302.23
3,1388835738,"Sembritzki, Laura Charlotte","Penter, Tanja",„Nukleares Naturschutzgebiet“ im Süd-Ural: At...,NaN,Laura Charlotte Sembritzki ; Betreuer: Tanja P...,Universitätsbibliothek Heidelberg,Heidelberg,Online-Ressource,2026,https://nbn-resolving.org/urn:nbn:de:bsz:16-he...,360
4,1402082711,"Schuldt, Moritz Philip","Mastalerz, Michael",Towards Planar and Tubular Haeckelites,NaN,Moritz Philip Schuldt ; Betreuer: Michael Mast...,Universitätsbibliothek Heidelberg,Heidelberg,Online-Ressource,2026,https://doi.org/10.11588/heidok.00038801; http...,516
...,...,...,...,...,...,...,...,...,...,...,...,...
1166,1352746786,NaN,NaN,Technik in den Online-Hochschulschriften (Meta...,NaN,NaN,NaN,NaN,201382 Datensätze,NaN,https://nbn-resolving.org/urn:nbn:de:101:3-241...,600; 610; 620; 621.3; 624; 630; 640; 650; 660;...
1167,1154073645,NaN,"Kapr, Albert",Verleihungsurkunde Doctor Philosophiae,NaN,für Albert Kapr,NaN,Leipzig,"1 Blatt (Druck, Kopie)",1974,NaN,NaN
1168,1288047533,"Krebs, Inga",NaN,Versicherungswissenschaftliche Dissertationen ...,NaN,von Inga Krebs,NaN,NaN,Online-Ressource,NaN,https://doi.org/10.1007/s12297-011-0169-8; htt...,360
1169,140377532X,NaN,NaN,XIX. Dissertationen,NaN,NaN,NaN,NaN,Online-Ressource,NaN,https://doi.org/10.7767/zrgra.1990.107.general...,340


### Speichern

In [ ]:
df.to_csv("test.csv", index=False, encoding="utf-8")